## This notebook shows how to create the different types of views

In [1]:
%load_ext sql

##### Connecting to the Database

In [2]:
%sql sqlite:///chinook.db

'Connected: @chinook.db'

In [4]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

### Task One: Creating a Lookup view for surname, first name, title, and country for each employee called employee_view

In [5]:
%sql PRAGMA table_list;

 * sqlite:///chinook.db
Done.


schema,name,type,ncol,wr,strict
main,sqlite_stat1,table,3,0,0
main,playlist_track,table,2,0,0
main,media_types,table,2,0,0
main,invoices,table,9,0,0
main,playlists,table,2,0,0
main,genres,table,2,0,0
main,employees,table,15,0,0
main,tracks,table,9,0,0
main,customers,table,13,0,0
main,sqlite_schema,table,5,0,0


In [6]:
%%sql

SELECT
    *
FROM
    employees
LIMIT 5;

 * sqlite:///chinook.db
Done.


EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
1,Adams,Andrew,General Manager,None,1962-02-18 00:00:00,2002-08-14 00:00:00,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
2,Edwards,Nancy,Sales Manager,1,1958-12-08 00:00:00,2002-05-01 00:00:00,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
3,Peacock,Jane,Sales Support Agent,2,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
4,Park,Margaret,Sales Support Agent,2,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
5,Johnson,Steve,Sales Support Agent,2,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com


In [8]:
%%sql

CREATE VIEW Employee_View AS
SELECT
    lastname,
    firstname,
    title,
    country
FROM
    employees;

 * sqlite:///chinook.db
Done.


[]

In [9]:
%%sql

SELECT
    name
FROM
    sqlite_master
WHERE
    type = 'view';

 * sqlite:///chinook.db
Done.


name
Employee_View


In [10]:
%%sql

SELECT
    *
FROM
    employee_view;

 * sqlite:///chinook.db
Done.


LastName,FirstName,Title,Country
Adams,Andrew,General Manager,Canada
Edwards,Nancy,Sales Manager,Canada
Peacock,Jane,Sales Support Agent,Canada
Park,Margaret,Sales Support Agent,Canada
Johnson,Steve,Sales Support Agent,Canada
Mitchell,Michael,IT Manager,Canada
King,Robert,IT Staff,Canada
Callahan,Laura,IT Staff,Canada


### Task Two: Querying the employee_view to get a view of the sales team

In [11]:
%%sql

CREATE VIEW sales_team AS
SELECT
    lastname,
    firstname,
    title,
    country
FROM
    employees
WHERE
    title LIKE 'Sales%';

 * sqlite:///chinook.db
Done.


[]

In [12]:
%%sql

SELECT
    *
FROM
    sales_team;

 * sqlite:///chinook.db
Done.


LastName,FirstName,Title,Country
Edwards,Nancy,Sales Manager,Canada
Peacock,Jane,Sales Support Agent,Canada
Park,Margaret,Sales Support Agent,Canada
Johnson,Steve,Sales Support Agent,Canada


### Task Three: Creating a Join View linking customers with their supporting staff

In [14]:
%%sql

SELECT
    *
FROM
    customers
LIMIT 5;

 * sqlite:///chinook.db
Done.


CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
2,Leonie,Köhler,None,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,+49 0711 2842222,None,leonekohler@surfeu.de,5
3,François,Tremblay,None,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,None,ftremblay@gmail.com,3
4,Bjørn,Hansen,None,Ullevålsveien 14,Oslo,None,Norway,0171,+47 22 44 22 22,None,bjorn.hansen@yahoo.no,4
5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,None,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


In [95]:
%%sql

CREATE VIEW customer_support_view AS 
SELECT
    cust.firstname AS Customer_firstname,
    cust.lastname AS Customer_lastname,
    cust.country AS customer_country,
    emp.employee_id,
    emp.employee_lastname,
    emp.employee_firstname,
    emp.title,
    emp.employee_country
FROM
    customers AS cust
INNER JOIN
    (
        SELECT
            employeeid AS employee_id,
            lastname AS employee_lastname,
            firstname AS employee_firstname,
            title,
            country AS employee_country
        FROM
            employees
        
    ) AS emp
ON
    cust.supportrepid = emp.employee_id;

 * sqlite:///chinook.db
Done.


[]

In [94]:
%sql DROP VIEW customer_support_view;

 * sqlite:///chinook.db
Done.


[]

In [96]:
%%sql

SELECT
    *
FROM
    customer_support_view;

 * sqlite:///chinook.db
Done.


Customer_firstname,Customer_lastname,customer_country,employee_id,employee_lastname,employee_firstname,title,employee_country
Luís,Gonçalves,Brazil,3,Peacock,Jane,Sales Support Agent,Canada
Leonie,Köhler,Germany,5,Johnson,Steve,Sales Support Agent,Canada
François,Tremblay,Canada,3,Peacock,Jane,Sales Support Agent,Canada
Bjørn,Hansen,Norway,4,Park,Margaret,Sales Support Agent,Canada
František,Wichterlová,Czech Republic,4,Park,Margaret,Sales Support Agent,Canada
Helena,Holý,Czech Republic,5,Johnson,Steve,Sales Support Agent,Canada
Astrid,Gruber,Austria,5,Johnson,Steve,Sales Support Agent,Canada
Daan,Peeters,Belgium,4,Park,Margaret,Sales Support Agent,Canada
Kara,Nielsen,Denmark,4,Park,Margaret,Sales Support Agent,Canada
Eduardo,Martins,Brazil,4,Park,Margaret,Sales Support Agent,Canada


### Task Four: Querying the customer_support_view to obtain a list of names and surnames of clients helped by employee id=3

In [63]:
%%sql

SELECT
    customer_lastname,
    customer_firstname
FROM
    customer_support_view
WHERE
    employee_id = 3;

 * sqlite:///chinook.db
Done.


Customer_lastname,Customer_firstname
Gonçalves,Luís
Tremblay,François
Almeida,Roberto
Peterson,Jennifer
Brooks,Michelle
Goyer,Tim
Ralston,Frank
Brown,Robert
Francis,Edward
Sullivan,Ellie


### Task Five: Creating an aggregating view that counts the number of customers currently being serviced per country

In [73]:
%%sql

CREATE VIEW customer_per_country_view AS
SELECT
    customer_country,
    COUNT(customer_firstname) AS Number_of_customers
FROM
    customer_support_view
GROUP BY
    customer_country;

 * sqlite:///chinook.db
Done.


[]

In [74]:
%%sql

SELECT
    *
FROM
    customer_per_country_view;

 * sqlite:///chinook.db
Done.


customer_country,Number_of_customers
Argentina,1
Australia,1
Austria,1
Belgium,1
Brazil,5
Canada,8
Chile,1
Czech Republic,2
Denmark,1
Finland,1


In [72]:
%sql DROP VIEW customer_per_country_view;

 * sqlite:///chinook.db
Done.


[]

### Task Six: Returning the country with the most customers from customer_per_country_view

In [77]:
%%sql

SELECT
    customer_country,
    MAX(number_of_customers)
FROM
    customer_per_country_view;

 * sqlite:///chinook.db
Done.


customer_country,MAX(number_of_customers)
USA,13


In [78]:
%%sql

SELECT
    name
FROM
    sqlite_master
WHERE
    type = 'view';

 * sqlite:///chinook.db
Done.


name
Employee_View
sales_team
customer_support_view
customer_per_country_view


In [79]:
%%sql

SELECT
    *
FROM
    employee_view;

 * sqlite:///chinook.db
Done.


LastName,FirstName,Title,Country
Adams,Andrew,General Manager,Canada
Edwards,Nancy,Sales Manager,Canada
Peacock,Jane,Sales Support Agent,Canada
Park,Margaret,Sales Support Agent,Canada
Johnson,Steve,Sales Support Agent,Canada
Mitchell,Michael,IT Manager,Canada
King,Robert,IT Staff,Canada
Callahan,Laura,IT Staff,Canada


### Task Seven: A query that returns the number of customers each support employee services , along with the name of the employee

In [80]:
%%sql

SELECT
    *
FROM
    customers
LIMIT 5;

 * sqlite:///chinook.db
Done.


CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
2,Leonie,Köhler,None,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,+49 0711 2842222,None,leonekohler@surfeu.de,5
3,François,Tremblay,None,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,None,ftremblay@gmail.com,3
4,Bjørn,Hansen,None,Ullevålsveien 14,Oslo,None,Norway,0171,+47 22 44 22 22,None,bjorn.hansen@yahoo.no,4
5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,None,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


In [82]:
%%sql

SELECT
    *
FROM
    employees
LIMIT 5;

 * sqlite:///chinook.db
Done.


EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
1,Adams,Andrew,General Manager,None,1962-02-18 00:00:00,2002-08-14 00:00:00,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
2,Edwards,Nancy,Sales Manager,1,1958-12-08 00:00:00,2002-05-01 00:00:00,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
3,Peacock,Jane,Sales Support Agent,2,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
4,Park,Margaret,Sales Support Agent,2,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
5,Johnson,Steve,Sales Support Agent,2,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com


In [92]:
%%sql

CREATE VIEW support_person_stats AS
SELECT
    emp.employeeid,
    COUNT(cust.firstname) AS Number_of_customers
FROM
    customers AS cust
INNER JOIN
    employees AS emp
ON
    emp.employeeid = cust.supportrepid
GROUP BY
    emp.employeeid;

 * sqlite:///chinook.db
Done.


[]

In [93]:
%%sql

SELECT
    *
FROM
    support_person_stats;

 * sqlite:///chinook.db
Done.


EmployeeId,Number_of_customers
3,21
4,20
5,18


In [91]:
%sql DROP VIEW support_person_stats;

 * sqlite:///chinook.db
Done.


[]